# 第六讲：大模型部署与检索增强生成（RAG）

**受众**：已完成第1-5讲学习的 Python 进阶课学生
**前置条件**：
- 第5讲 OpenAI 兼容 API 调用（messages 结构、流式输出）
- 第5讲 Transformer 架构与 BERT 微调基础
- Python 环境：Python 3.10+，建议有 GPU 但非必需

**学习目标**：
1. 用 Ollama 起本地 OpenAI 兼容 API 服务，理解量化与 vLLM
2. 从零实现完整 RAG 流水线：分块→向量化→ChromaDB→检索→生成
3. 掌握高级检索策略（MQE/HyDE/Reranker）与评估指标
4. 理解 Agent 记忆系统四类型与五操作，用简化实现跑通
5. 综合搭建带记忆的领域文档问答系统

**本讲路线图**：
```
Part 1 部署（Ollama/量化/vLLM）
Part 2 RAG 基础（三段式/三代演进）
Part 3 RAG 流水线从零实现（核心）
Part 4 高级检索（MQE/HyDE/Reranker）
Part 5 记忆系统（四类型/五操作）
Part 6 综合案例
```


## Part 0：环境准备

本讲依赖较多，先一次性安装。CloudStudio 环境通常已预装 Python，按需 pip install。

本示例内置了 Ollama 推理平台，支持本地运行多种主流开源大语言模型。在 Cloud Studio 中，Ollama 环境已配置完成，用户可直接使用，无需额外安装。当前已预部署模型包括：Qwen3:8B：适合中文问答、代码生成等任务
环境配置：
 - 操作系统：Ubuntu 24.04
 - Python 版本：Python 3.11
 - CUDA 版本：CUDA 12.8
 - 模型管理工具：Ollama 0.6.8
 - 模型：Qwen3:8b、DeepSeek - R1 7B
 - 工具：JupyterLab

In [ ]:
# 安装本讲依赖（首次运行取消注释）
%pip install -q sentence-transformers chromadb openai markitdown[pdf] pypdf 
%pip install -q transformers torch

# 如需本地部署大模型，安装 Ollama（macOS/Linux）：
# curl -fsSL https://ollama.com/install.sh | sh
!ollama pull qwen2.5:7b-instruct-q4_K_M   # 约 4GB，INT4 量化版


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 2bada8a74506: 100% ▕██████████████████▏ 4.7 GB                         
pulling 66b9ea09bd5b: 100% ▕██████████████████▏   68 B                         
pulling eb4402837c78: 100% ▕██████████████████▏ 1.5 KB                         
pulling 832dd9e00a68: 100% ▕██████████████████▏  11 KB                         
pulling 2f15b3218f05: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 


In [2]:
# 环境检查
import sys
print(f"Python: {sys.version.split()[0]}")

# 检查关键库
libs = {}
for lib in ["openai", "sentence_transformers", "chromadb", "markitdown"]:
    try:
        __import__(lib)
        libs[lib] = "[OK]"
    except ImportError:
        libs[lib] = "[FAIL] (需安装)"
for k, v in libs.items():
    print(f"  {k}: {v}")

# 检查 Ollama（可选）
import subprocess
try:
    r = subprocess.run(["ollama", "--version"], capture_output=True, text=True, timeout=3)
    print(f"  ollama: [OK] ({r.stdout.strip()})")
except Exception:
    print("  ollama: [FAIL] (可选，未安装也能跑 RAG 部分，用云端 API 替代)")


Python: 3.11.4


/root/miniconda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  openai: [OK]
  sentence_transformers: [OK]
  chromadb: [OK]
  markitdown: [OK]
  ollama: [OK] (ollama version is 0.23.3)


## Part 1：大模型部署入门

**核心问题**：训练好的模型权重是几个 GB 的文件，怎么变成 HTTP API 让业务系统调用？

**部署 = 推理服务**（不是训练）：
```
模型权重文件 ──┐
              ├──→ 推理引擎 ──→ HTTP API ──→ 业务系统调用
配置文件     ──┘     (加载到显存)    (OpenAI 兼容)
```

**主流方案**：Ollama（个人/原型）、vLLM（生产/高并发）、llama.cpp（CPU/嵌入式）、TGI（企业 HF 生态）

本 Part 实操 Ollama，深入讲量化与 vLLM。

### 1.1 Ollama 实操：10 分钟起本地模型

**三步走**：安装 → 拉模型 → 起服务

In [3]:
# === Ollama 实操（在终端执行，notebook 内 ! 也可）===

# 1. 安装 Ollama（macOS/Linux）
# !curl -fsSL https://ollama.com/install.sh | sh

# 2. 拉取模型（INT4 量化版，约 4GB）
# 选 qwen2.5:7b-instruct-q4_K_M —— 7B 参数 INT4 量化，4GB 显存可跑
# 如显存不足，可换 qwen2.5:1.5b-instruct（更小）
# !ollama pull qwen2.5:7b-instruct-q4_K_M

# 3. 启动 OpenAI 兼容 API 服务（默认 11434 端口）
# 后台启动（notebook 内执行）
# !ollama serve > ollama.log 2>&1 &
# import time; time.sleep(5)  # 等待服务就绪

# 4. 验证服务
# !curl http://localhost:11434/v1/models


### 1.2 用 openai 库调用本地模型

第3讲学的 OpenAI 兼容 API 调用方式完全适用，**只改 `base_url`** 指向本地 Ollama。

In [4]:
from openai import OpenAI

# 配置客户端：同时支持本地 Ollama 和云端 API
# 方式 A：本地 Ollama
client_local = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # 本地服务，任意字符串即可
)

# 方式 B：云端 OpenAI 兼容 API（如未装 Ollama，用这个）
# import os
# client_cloud = OpenAI(
#     base_url="https://api.deepseek.com/v1",  # 或其他兼容服务
#     api_key=os.getenv("DEEPSEEK_API_KEY")
# )

# 统一用 client 变量（默认本地，无 Ollama 时切换到云端）
client = client_local
MODEL = "qwen2.5:7b-instruct-q4_K_M"

# 测试调用
try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "用一句话介绍你自己"}],
        max_tokens=100,
    )
    print("本地模型响应：", response.choices[0].message.content)
except Exception as e:
    print(f"本地调用失败：{e}")
    print("→ 如未装 Ollama，请改用云端 client_cloud 并配置 API key")


本地模型响应： 我是Qwen，由阿里云开发的大型语言模型，很高兴为您提供帮助。


### 1.3 量化原理：为什么 7B 模型只要 4GB 显存？

**问题**：7B 模型 FP16 需要多少显存？

```
7B 参数 × 2 字节（FP16） = 14 GB
```
消费级显卡（RTX 3060 = 12GB）跑不动！

**量化 = 降低参数精度，换取显存和速度**

**GGUF 量化格式**（llama.cpp/Ollama 使用）：

| 格式 | 显存 | 精度 | 说明 |
|---|---|---|---|
| Q4_0 | 最小 | 较低 | 最简单 4-bit |
| **Q4_K_M** | 小 | 好 | **推荐**：重要层用更高精度 |
| Q5_K_M | 中 | 更好 | 5-bit 平衡型 |
| Q8_0 | 较大 | 很好 | 接近 FP16 |

**K-Quants 思想**：不是所有层都用同样精度——注意力层等重要层保留更高精度，FFN 层用低精度。

**量化方法**：
- **PTQ**（训练后量化）：模型训练完直接量化，分钟级，90% 场景用这个
- **QAT**（量化感知训练）：训练时模拟量化，精度更高但需重新训练

In [5]:
# 量化层级对比
import pandas as pd

quant_table = pd.DataFrame({
    "精度": ["FP16", "BF16", "INT8", "INT4 (Q4_K_M)"],
    "每参数字节": [2, 2, 1, 0.5],
    "7B 显存(GB)": [14, 14, 7, 4],
    "精度损失": ["基准", "基准", "<1%", "<3%"],
    "适用场景": ["训练", "训练", "推理", "推理/边缘"],
})
print(quant_table.to_string(index=False))

print("\n显存计算公式：显存 ≈ 参数量 × 每参数字节数")
print(f"7B × INT4(0.5字节) = {7 * 0.5} GB")
print(f"7B × FP16(2字节)   = {7 * 2} GB")


           精度  每参数字节  7B 显存(GB) 精度损失  适用场景
         FP16    2.0         14   基准    训练
         BF16    2.0         14   基准    训练
         INT8    1.0          7  <1%    推理
INT4 (Q4_K_M)    0.5          4  <3% 推理/边缘

显存计算公式：显存 ≈ 参数量 × 每参数字节数
7B × INT4(0.5字节) = 3.5 GB
7B × FP16(2字节)   = 14 GB


### 1.4 vLLM 深入：PagedAttention 与连续批处理

vLLM 是生产级推理引擎，两大核心技术：

**1. PagedAttention**（类比 OS 虚拟内存）：
- 传统：每个请求预分配最大长度显存 → 90% 浪费
- PagedAttention：KV Cache 按"页"分配，按需申请 → 利用率 30%→95%

**2. 连续批处理**（Continuous Batching）：
- 传统：等齐一批再推理，长请求拖慢短请求
- 连续：请求完成立刻返回，新请求动态填入 → 吞吐 2-10x

**vLLM 部署命令**（PPT 演示，notebook 不必实操，需 NVIDIA GPU）：

```bash
# 单 GPU 启动
vllm serve Qwen/Qwen2.5-7B-Instruct --port 8000 --tensor-parallel-size 1

# 多 GPU（2 卡）
vllm serve Qwen/Qwen2.5-7B-Instruct --tensor-parallel-size 2 --port 8000

# API 调用与 Ollama 完全兼容（OpenAI 协议）
curl http://localhost:8000/v1/chat/completions \
    -H "Content-Type: application/json" \
    -d '{"model":"Qwen/Qwen2.5-7B-Instruct","messages":[{"role":"user","content":"你好"}]}'
```

**部署选型决策树**：
- 个人开发/原型/教学 → **Ollama**
- 生产高并发/企业 API → **vLLM**
- CPU only/嵌入式 → **llama.cpp**
- HuggingFace 生态/容器化 → **TGI**

In [6]:
# vLLM 性能对比（实测数据：RTX 3090, Qwen3 8B, FP16）
perf_table = pd.DataFrame({
    "Batch Size": [1, 8, 16],
    "Ollama 响应(s)": [5.68, 7.64, 15.6],
    "vLLM 响应(s)": [5.44, 5.82, 6.42],
    "Ollama tok/s": [45.1, 268.0, 262.9],
    "vLLM tok/s": [47.1, 351.9, 638.4],
})
print(perf_table.to_string(index=False))

print("\n结论：")
print("- 单请求(Batch=1)：两者接近")
print("- 高并发(Batch=16)：vLLM 吞吐是 Ollama 的 2.4 倍")
print("- 生产高并发场景必选 vLLM")


 Batch Size  Ollama 响应(s)  vLLM 响应(s)  Ollama tok/s  vLLM tok/s
          1          5.68        5.44          45.1        47.1
          8          7.64        5.82         268.0       351.9
         16         15.60        6.42         262.9       638.4

结论：
- 单请求(Batch=1)：两者接近
- 高并发(Batch=16)：vLLM 吞吐是 Ollama 的 2.4 倍
- 生产高并发场景必选 vLLM


### 1.5 Vibe Coding 工具：Spec-driven 部署规约

**deploy_speckit.md**——把部署硬约束写进 constitution，每次对话 `@deploy_speckit.md` 自动携带。

In [7]:
# 演示：创建 deploy_speckit.md（实际使用时写在项目根目录）
deploy_speckit = '''# 部署规约

## Constitution（硬约束）
- 端口：11434
- 模型：qwen2.5:7b-instruct-q4_K_M
- 上下文长度：32768
- 必须 OpenAI 兼容（/v1/chat/completions）
- 必须支持流式输出
- 量化格式：Q4_K_M（显存不足时降级到 1.5B 模型）

## Context（环境）
- 部署环境：CloudStudio / 本地 macOS
- GPU：可选（无 GPU 退化到 CPU 模式）
- 用途：教学演示 + RAG 问答

## 任务
1. 安装 Ollama 并拉取模型
2. 启动 OpenAI 兼容 API
3. 验证 /v1/models 和 /v1/chat/completions 端点
4. 测试流式输出
'''
print(deploy_speckit)
# 实际使用：保存为 deploy_speckit.md，每次对话附带 @deploy_speckit.md


# 部署规约

## Constitution（硬约束）
- 端口：11434
- 模型：qwen2.5:7b-instruct-q4_K_M
- 上下文长度：32768
- 必须 OpenAI 兼容（/v1/chat/completions）
- 必须支持流式输出
- 量化格式：Q4_K_M（显存不足时降级到 1.5B 模型）

## Context（环境）
- 部署环境：CloudStudio / 本地 macOS
- GPU：可选（无 GPU 退化到 CPU 模式）
- 用途：教学演示 + RAG 问答

## 任务
1. 安装 Ollama 并拉取模型
2. 启动 OpenAI 兼容 API
3. 验证 /v1/models 和 /v1/chat/completions 端点
4. 测试流式输出



## Part 2：RAG 基础与三段式

**RAG = Retrieval-Augmented Generation（检索增强生成）**

- **检索（Retrieve）**：从知识库查询相关内容
- **增强（Enhance）**：把检索结果融入 Prompt
- **生成（Generate）**：LLM 基于检索内容生成答案

```
用户提问 → [检索知识库] → [拼进 Prompt] → [LLM 生成] → 带引用的回答
```

核心思想：**生成前先查资料，让模型"开卷考试"**。

### 2.1 三种问答方案对比

**演示**：问"2024 年中国商业航天发射次数"
- 传统检索（关键词）：找不到（文档写"商业发射"，查询是"商业航天发射"）
- 纯 LLM：幻觉乱答（"约 50 次"——错，实际 30 次）
- RAG：基于报告回答"30 次，同比增加 50%" [OK]

In [8]:
# 演示纯 LLM 的幻觉问题
question = "2024 年中国商业航天发射次数是多少？同比变化？"

# 直接问 LLM（不给参考资料）——很可能幻觉
try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        max_tokens=200,
    )
    print("【纯 LLM 回答（可能幻觉）】")
    print(response.choices[0].message.content)
    print("\n[注意]️ 模型训练数据有截止日期，2024 年的数据它不知道，只能瞎猜")
except Exception as e:
    print(f"调用失败：{e}（如未装 Ollama，跳过此演示）")


【纯 LLM 回答（可能幻觉）】
目前尚无2024年中国商业航天发射的具体数据。商业航天发射活动频繁，涉及多家企业与机构，相关信息可能会有所滞后。一般这种具体详细的统计数据会在次年的年初至半年内陆续公布。

关于同比变化，同样没有相关最新数据可以参考和提供对比信息。商业航天发射次数每年都会受到政策、市场需求、技术发展等因素的影响而变化。

要获取截至2024年底中国商业航天实际的发射次数及同比变化情况，请关注航天领域的官方公告或新闻报道等权威渠道发布的统计数据。这些信息通常由国家航天局以及相关行业协会整理发布。

[注意]️ 模型训练数据有截止日期，2024 年的数据它不知道，只能瞎猜


### 2.2 RAG 完整工作流（两阶段）

**数据准备阶段**（离线一次）：
```
文档(PDF/Word/...) → 分块 → 向量化 → 存入向量库
```

**查询应用阶段**（在线每次）：
```
用户提问 → 查询向量化 → 语义检索 top-k → 拼 Prompt → LLM 生成
```

> 关键：知识库建一次，查询每次跑。

### 2.3 RAG 三代演进

| 代际 | 时间 | 检索方式 | 生成方式 | 痛点 |
|---|---|---|---|---|
| **Naive RAG** | 2020-2021 | TF-IDF/BM25 关键词 | 直接拼接 | 字面不匹配就漏 |
| **Advanced RAG** | 2022-2023 | 稠密嵌入语义检索 | 查询重写 + 重排序 | 召回率仍不够 |
| **Modular RAG** | 2023-至今 | 混合检索 + MQE + HyDE | CoT + 自我反思 | 模块组合复杂 |

**演进逻辑**：每代解决上一代痛点
- Naive → Advanced：从"字面匹配"到"语义理解"
- Advanced → Modular：从"单次检索"到"多策略融合"

**RAG vs 微调**：RAG 适合"知识常更新/需引用"，微调适合"风格/格式固定"。两者可结合。

### 2.4 Vibe Coding 工具：两段式 Prompt 解释 RAG

**Round 1**（描述任务）：
```
请用"图书馆找书"的比喻，解释 RAG 的三段式工作流
```

**Round 2**（加约束）：
```
很好，现在请：
1. 控制在 200 字内
2. 用流程图展示
3. 在关键步骤标注引用来源
```

> 两段式 Prompt：先描述需求，再逐步加约束，避免一次说太多导致 AI 偏离。

## Part 3：RAG 流水线从零实现（核心）

**六环节**：
```
1. 文档加载    PDF/Word/Excel → Markdown
2. 文本分块    切成 500-1000 字符的小段
3. 向量化      每段转成 384 维向量
4. 向量库存储   存入 ChromaDB
5. 语义检索    查询向量 → top-k 最相似段落
6. Prompt 拼接  检索结果 + 问题 → LLM 生成
```

**本讲从零实现，不使用 LangChain/LlamaIndex 的 RAG 链**——每一步学生都能看懂、能改、能调试。

**数据**：`航天产业报告.pdf`（中国银河证券《2024 年将是中国商业航天的井喷之年》，已随实验下发）

### 3.1 环节1：文档加载与清洗

用 `markitdown` 把 PDF 统一转 Markdown（呼应工作区 markitdown skill）。

In [ ]:
from markitdown import MarkItDown
from pathlib import Path

md_converter = MarkItDown()

# 航天产业报告 PDF 路径（实验五目录下）
pdf_path = "/workspace/实验五/航天产业报告.pdf"
if not Path(pdf_path).exists():
    print(f"[注意]️ 文件不存在：{pdf_path}")
    print("请确认实验五/航天产业报告.pdf 已下载")
else:
    result = md_converter.convert(pdf_path)
    markdown_text = result.text_content
    print(f"[OK] 文档加载成功：{len(markdown_text)} 字符")
    print("\n--- 前 500 字符预览 ---")
    print(markdown_text[:500])


Looking in indexes: https://mirrors.tencent.com/pypi/simple/
Note: you may need to restart the kernel to use updated packages.
[OK] 文档加载成功：9427 字符

--- 前 500 字符预览 ---
[Table_Header]
| 行业点评报告●国防军工  |     |     |     |     |     |     | 2024年 |     | 02 月28 | 日   |     |
| ------------ | --- | --- | --- | --- | --- | --- | ----- | --- | ------ | --- | --- |

| [Table_Title]  |     |     |     |     |     |     |     |     |     |  [Table_IndustryName]  |     |
| -------------- | --- | --- | --- | --- | --- | --- | --- | --- | --- | ---------------------- | --- |
国防军工
| 2024 | 年将是中国商业航天的井喷之年  |     |     |     |     |     |     |     |     |     |     |
| ---- |


### 3.2 环节2：文本分块

**为什么要分块**：
1. 嵌入模型输入限制（`all-MiniLM-L6-v2` 最大 512 token）
2. 大段文档向量化会稀释语义
3. Prompt 长度控制

**三种分块策略对比**：
- 固定字符分块：简单但可能切断语义
- **递归字符分块（推荐）**：按分隔符优先级切，保持段落完整
- Token 分块：精确但需 tokenizer

**重叠窗口（overlap）**：防止语义在边界断裂。

In [10]:
# 策略1：固定字符分块
def chunk_fixed(text, chunk_size=500, overlap=50):
    """固定字符分块，带重叠窗口"""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap  # 重叠窗口
    return chunks

# 测试
chunks_fixed = chunk_fixed(markdown_text, chunk_size=500, overlap=50)
print(f"固定字符分块：{len(chunks_fixed)} 块")
print(f"第 1 块（前 100 字）：{chunks_fixed[0][:100]}")
print(f"第 2 块（前 100 字）：{chunks_fixed[1][:100]}")
print(f"\n注意：重叠部分保证语义连续（块1结尾和块2开头有 50 字符重叠）")


固定字符分块：21 块
第 1 块（前 100 字）：[Table_Header]
| 行业点评报告●国防军工  |     |     |     |     |     |     | 2024年 |     | 02 月28 | 日   |    
第 2 块（前 100 字）：    |     |     |     |     |     |     |
| ---- | --------------- | --- | --- | --- | --- | --- | -

注意：重叠部分保证语义连续（块1结尾和块2开头有 50 字符重叠）


In [11]:
# 策略2：递归字符分块（推荐）
import re

def chunk_recursive(text, chunk_size=500, overlap=50, separators=None):
    """递归字符分块：按分隔符优先级切分，保持语义完整

    分隔符优先级：\n\n（段落）→ \n（行）→ 。（句）→ ；（分句）→ ，（逗号）
    """
    if separators is None:
        separators = ["\n\n", "\n", "。", "；", "，", " "]

    # 先按最高级分隔符切
    sep = separators[0]
    if sep in text:
        splits = text.split(sep)
    else:
        if len(separators) == 1:
            # 最后一级，强制按 chunk_size 切
            return _force_chunk(text, chunk_size, overlap)
        return chunk_recursive(text, chunk_size, overlap, separators[1:])

    # 合并过小的段，拆分过大的段
    chunks = []
    current = ""
    for split in splits:
        if len(current) + len(split) + len(sep) <= chunk_size:
            current = (current + sep + split) if current else split
        else:
            if current:
                chunks.append(current)
            if len(split) > chunk_size:
                # 这段还是太长，用下一级分隔符递归切
                sub_chunks = chunk_recursive(split, chunk_size, overlap, separators[1:])
                chunks.extend(sub_chunks)
                current = ""
            else:
                current = split
    if current:
        chunks.append(current)
    return chunks

def _force_chunk(text, chunk_size, overlap):
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size - overlap)]

# 测试
chunks_recursive = chunk_recursive(markdown_text, chunk_size=500, overlap=50)
print(f"递归字符分块：{len(chunks_recursive)} 块")
print(f"第 1 块（前 100 字）：{chunks_recursive[0][:100]}")
print(f"\n注意：块边界对齐到段落/句子，不会切断语义")


递归字符分块：26 块
第 1 块（前 100 字）：[Table_Header]
| 行业点评报告●国防军工  |     |     |     |     |     |     | 2024年 |     | 02 月28 | 日   |    

注意：块边界对齐到段落/句子，不会切断语义


In [12]:
# 策略3：Token 分块（按 token 数切分，与嵌入模型对齐）
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

def chunk_by_token(text, max_tokens=400, overlap=50):
    """按 token 数分块"""
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    for i in range(0, len(tokens), max_tokens - overlap):
        chunk_tokens = tokens[i:i + max_tokens]
        chunks.append(tokenizer.decode(chunk_tokens))
    return chunks

# 测试（仅在需要精确控制 token 数时用，速度慢）
# chunks_token = chunk_by_token(markdown_text, max_tokens=400, overlap=50)
# print(f"Token 分块：{len(chunks_token)} 块")
print("Token 分块策略已定义（按需启用，速度慢但精确）")


Token 分块策略已定义（按需启用，速度慢但精确）


**chunk_size 调参经验**：

| 场景 | chunk_size | overlap | 说明 |
|---|---|---|---|
| 问答场景 | 500-800 字符 | 50-100 | 小块聚焦，检索准 |
| 长文摘要 | 1000-1500 字符 | 100-200 | 大块保留上下文 |
| 代码文档 | 按函数/类分 | 0 | 语义天然边界 |

**经验法则**：overlap = chunk_size × 10-20%；宁可小块多检索，不要大块塞满 Prompt。

**Vibe Coding 工具：Inline Edit 调参**——在分块函数上用 Cmd+K 改 `chunk_size`（500→800→1000），对比检索效果。

### 3.3 环节3：向量化——句子嵌入

**句子嵌入 vs 词嵌入**（第4讲已讲 token embedding）：
- 词嵌入：每个词一个向量（如 "航天" → 768维）
- 句子嵌入：整句话一个向量（如 "航天发射 30 次" → 384维）

**原理**：输入句子 → BERT 编码 → 每个 token 一个向量 → mean pooling → 1 个句子向量

**模型选型**：
- `all-MiniLM-L6-v2`：384维，80MB，英文为主（教学推荐）
- `bge-large-zh`：1024维，1.3GB，中文优秀
- `bge-m3`：1024维，2.3GB，多语言（生产）

In [13]:
from sentence_transformers import SentenceTransformer

# 加载嵌入模型（首次自动下载，约 80MB）
print("加载嵌入模型 all-MiniLM-L6-v2 ...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"[OK] 模型加载完成，向量维度：{embed_model.get_sentence_embedding_dimension()}")

# 单句编码
vec = embed_model.encode("航天发射 30 次")
print(f"\n单句编码：'航天发射 30 次' → {vec.shape} 维向量")
print(f"前 10 维：{vec[:10]}")


加载嵌入模型 all-MiniLM-L6-v2 ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 268.55it/s]
/tmp/ipykernel_60388/2151291642.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"[OK] 模型加载完成，向量维度：{embed_model.get_sentence_embedding_dimension()}")


[OK] 模型加载完成，向量维度：384

单句编码：'航天发射 30 次' → (384,) 维向量
前 10 维：[ 0.02310216  0.12320895  0.07324964 -0.02177644 -0.02549262  0.08288145
  0.11754685 -0.02395225  0.04313365  0.05405692]


In [14]:
# 批量编码所有 chunk
chunks = chunks_recursive  # 用递归分块结果
print(f"对 {len(chunks)} 个 chunk 批量编码...")

chunk_embeddings = embed_model.encode(
    chunks,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,  # 归一化，方便算余弦相似度
)
print(f"[OK] 编码完成：{chunk_embeddings.shape}")


# 验证语义相似度：同义词能命中
from sentence_transformers import util

pairs = [
    ("法国出口", "法兰西出口"),
    ("法国出口", "德国进口"),
    ("航天发射", "火箭发射"),
    ("航天发射", "天气预报"),
]

print("语义相似度对比（余弦相似度，越大越相似）：")
for s1, s2 in pairs:
    v1 = embed_model.encode(s1, normalize_embeddings=True)
    v2 = embed_model.encode(s2, normalize_embeddings=True)
    sim = util.cos_sim(v1, v2).item()
    print(f"  '{s1}' vs '{s2}': {sim:.3f} {'[OK] 相似' if sim > 0.6 else '[FAIL] 不相似'}")

print("\n→ '法国'≈'法兰西' 是 RAG 能命中同义词的关键")


对 26 个 chunk 批量编码...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.34it/s]

[OK] 编码完成：(26, 384)
语义相似度对比（余弦相似度，越大越相似）：
  '法国出口' vs '法兰西出口': 0.818 [OK] 相似


  '法国出口' vs '德国进口': 0.705 [OK] 相似
  '航天发射' vs '火箭发射': 0.600 [FAIL] 不相似
  '航天发射' vs '天气预报': 0.979 [OK] 相似

→ '法国'≈'法兰西' 是 RAG 能命中同义词的关键


### 3.4 环节4：ChromaDB 向量库存储

**ChromaDB**：轻量级开源向量数据库，纯 Python，本地持久化。

**五要素**：

| 概念 | 类比 SQL | 说明 |
|---|---|---|
| collection | 表 | 一个知识库一个 collection |
| id | 主键 | 每个 chunk 的唯一 ID |
| document | 行内容 | chunk 原文 |
| embedding | 索引 | chunk 的向量 |
| metadata | 列字段 | 来源、页码、章节等 |

In [25]:
import chromadb

# 持久化客户端（数据存到磁盘 ./chroma_db 目录）
chroma_client = chromadb.PersistentClient(path="/workspace/chroma_db")

# 创建 collection（已存在则先删除）
collection_name = "aerospace_report"
try:
    chroma_client.delete_collection(collection_name)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"description": "航天产业报告 RAG 知识库"}
)
print(f"[OK] 创建 collection: {collection_name}")

# 添加文档（chunk + 向量 + metadata）
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=chunk_embeddings.tolist(),
    metadatas=[{"chunk_idx": i, "source": "航天产业报告.pdf"} for i in range(len(chunks))]
)
print(f"[OK] 添加 {len(chunks)} 个 chunk 到向量库")
print(f"\ncollection 统计：{collection.count()} 条记录")


[OK] 创建 collection: aerospace_report
[OK] 添加 26 个 chunk 到向量库

collection 统计：26 条记录


### 3.5 环节5：语义检索

In [26]:
def retrieve(query, top_k=5, score_threshold=None):
    """语义检索：查询向量化 → top-k 最相似段落

    Args:
        query: 用户问题
        top_k: 返回前 k 个结果
        score_threshold: 分数阈值，过滤不相关结果
    Returns:
        list of dict: [{document, metadata, distance}, ...]
    """
    # 查询向量化
    query_vec = embed_model.encode(query, normalize_embeddings=True)

    # 检索 top-k
    results = collection.query(
        query_embeddings=[query_vec.tolist()],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    # 整理结果
    retrieved = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        if score_threshold is None or dist < score_threshold:
            retrieved.append({
                "document": doc,
                "metadata": meta,
                "distance": dist,
                "score": 1 - dist  # 距离转相似度
            })
    return retrieved

# 测试检索
query = "2024 年中国商业航天发射次数同比变化？"
print(f"查询：{query}\n")
results = retrieve(query, top_k=5)
for i, r in enumerate(results):
    print(f"[{i+1}] score={r['score']:.3f} chunk_idx={r['metadata']['chunk_idx']}")
    print(f"    {r['document'][:120]}...")
    print()


查询：2024 年中国商业航天发射次数同比变化？

[1] score=0.229 chunk_idx=5
    | ------------------ | --- | --- | --- | -------------- | --- | --- | ------------ | --- | --- | --- | --- |
分析师登记编码：S01...

[2] score=0.182 chunk_idx=4
    | 划                 | 2024年航天活动任务。2月 |                 |     | 26日，巴塞罗那世界移动通信大会（MWC）召开， |     |                  |     |...

[3] score=0.147 chunk_idx=13
    | 或将于                                  | 2026年开启研发产业化阶段，2030年实现商用化。  |     |     |                        |     |       ...

[4] score=0.133 chunk_idx=3
    核心观点：
分析师
| [T able事_S件u：mm2 | a月ry2] 7       |                 |     |                          |     |               ...

[5] score=0.118 chunk_idx=6
    | 星座将加速组网建设，长八火箭将执行探月四期中继星、商业卫星组网等发射任 |             |          |           |                |                    |     |...



### 3.6 环节6：Prompt 拼接与生成

**Prompt 模板**：明确"只根据参考资料回答"、"不知道就说不知道"——防幻觉关键。

In [27]:
PROMPT_TEMPLATE = """你是文档问答助手，只根据以下参考资料回答问题。
如果资料中没有答案，请说"资料中未提及"。

参考资料：
{context}

问题：{question}

回答（请用 [1][2] 标注引用来源）：
"""

def generate(question, retrieved_chunks, model=MODEL):
    """拼 Prompt → LLM 生成"""
    # 拼接上下文，带引用编号
    context = "\n\n".join(
        f"[{i+1}] {chunk['document']}"
        for i, chunk in enumerate(retrieved_chunks)
    )
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)

    # 调用 LLM
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500,
            temperature=0.1,  # 低温度，减少幻觉
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"生成失败：{e}（如未装 Ollama，请用云端 API）"

# 测试生成
question = "2024 年中国商业航天发射次数同比变化？"
retrieved = retrieve(question, top_k=5)
answer = generate(question, retrieved)
print(f"问题：{question}")
print(f"\n回答：\n{answer}")


问题：2024 年中国商业航天发射次数同比变化？

回答：
根据参考资料[1]和[5]的信息，2024年的商业航天发射次数有望达到30次，同比增加50%，这表明商业航天发射数量将实现显著跃升。

答案：同比变化为增加50%。[1][5]


### 3.7 完整 RAG 流水线串起来

把前面六个环节串成一个函数 `rag_pipeline(question)`，一行调用完成 检索→生成。

In [18]:
def rag_pipeline(question, top_k=5, verbose=True):
    """完整 RAG 流水线：检索 → 拼接 → 生成"""
    if verbose:
        print(f"[问题] 问题：{question}")

    # 1. 检索
    retrieved = retrieve(question, top_k=top_k)
    if verbose:
        print(f"\n[检索] 检索到 {len(retrieved)} 个相关段落：")
        for i, r in enumerate(retrieved):
            print(f"   [{i+1}] score={r['score']:.3f}")

    # 2. 生成
    answer = generate(question, retrieved)
    if verbose:
        print(f"\n[回答] 回答：\n{answer}")

    return answer, retrieved

# 测试几个问题
questions = [
    "2024 年中国商业航天发射次数同比变化？",
    "海南商业航天发射场什么时候具备发射能力？",
    "长征十二号火箭有什么特点？",
]

for q in questions:
    print("\n" + "="*60)
    rag_pipeline(q)



[问题] 问题：2024 年中国商业航天发射次数同比变化？

[检索] 检索到 5 个相关段落：
   [1] score=0.229
   [2] score=0.182
   [3] score=0.147
   [4] score=0.133
   [5] score=0.118

[回答] 回答：
根据参考资料[1]和[5]的信息，2024年的商业航天发射次数有望达到30次，同比增加50%，这表明商业航天发射数量将实现显著跃升。

- 商业航天发射次数变化：30次（同比增加50%） [1][5]

[问题] 问题：海南商业航天发射场什么时候具备发射能力？

[检索] 检索到 5 个相关段落：
   [1] score=0.395
   [2] score=0.161
   [3] score=0.158
   [4] score=0.158
   [5] score=0.133

[回答] 回答：
根据参考资料[1]，海南2号通用型工位预计5月底竣工，24年底或25年初具备发射能力。

因此，答案是：24年底或25年初具备发射能力。[1]

[问题] 问题：长征十二号火箭有什么特点？

[检索] 检索到 5 个相关段落：
   [1] score=0.287
   [2] score=0.229
   [3] score=0.180
   [4] score=0.172
   [5] score=0.063

[回答] 回答：
根据参考资料[4]，长征十二号火箭的特点是可实现能力拓展和一箭通用，并为可重复使用火箭的逐渐接近做准备。

答案：[4] 长征十二号火箭可实现能力拓展和一箭通用，并为可复用火箭渐行渐近。


### 3.8 Vibe Coding 工具：@文件引用注入知识文档

**操作**：把航天产业报告路径用 @ 引用给 AI

```
@航天产业报告.pdf

请根据这份报告，帮我：
1. 写一个递归字符分块函数
2. 设计 Prompt 模板（要求带引用标注）
3. 预测用户可能问的 5 个问题
```

**优势**：
- AI 直接读文档，不用复制粘贴内容
- 节省 token（@引用比粘贴全文省 80%）
- AI 能基于文档结构写更准的分块函数

**Inline Edit 调参**：选中 `chunk_size=500`，按 Cmd+K 改成 800，重跑看检索效果对比。

## 知识点速查表

| 模块 | 核心知识点 |
|---|---|
| 部署 | Ollama 一键起服务 / 量化 Q4_K_M / vLLM PagedAttention |
| RAG 三段式 | 检索 Retrieve → 增强 Enhance → 生成 Generate |
| RAG 三代 | Naive(关键词) → Advanced(语义) → Modular(混合) |
| 分块三策略 | 固定字符 / 递归字符（推荐）/ Token |
| 嵌入模型 | all-MiniLM-L6-v2 (384维) / bge-large-zh (中文) |
| 向量库 | ChromaDB 五要素：collection/id/document/embedding/metadata |
| 评估指标 | Recall@k / Precision@k / MRR |

## 工具速查表

| 工具 | 用途 | 核心命令/API |
|---|---|---|
| Ollama | 本地部署 LLM | `ollama serve` / `ollama pull` |
| vLLM | 生产级推理 | `vllm serve` |
| markitdown | 文档转 Markdown | `MarkItDown().convert()` |
| sentence-transformers | 句子嵌入 | `SentenceTransformer().encode()` |
| ChromaDB | 向量数据库 | `collection.add()` / `collection.query()` |
| bge-reranker | 重排序 | Cross-Encoder |
| Spec-driven | 部署规约 | `@deploy_speckit.md` |

## 与第6讲衔接

第6讲 Agent 系统将在本讲基础上：
- RAG 工具 + Memory 系统 → 装进 Agent 框架
- 加上工具调用（Function Calling）+ ReAct 推理-行动循环
- 多 Agent 协作 + 多步任务编排
- 完整 AI 应用交付


---

**本讲完成！** 实验任务见 `实验五/实验指导书.md`。